# Minifigures Application Deployment to AWS EC2

This notebook automates the deployment of the Minifigures Docker application to AWS EC2. Run each cell in order.

**Prerequisites:**
- Docker installed locally
- AWS CLI configured
- SSH key pair (`fresca-lorenzo-key-pair.pem`) available
- EC2 instance running with IP: `35.180.39.26`

## Step 1: Build Docker Image Locally

Run this command to build the Docker image with no cache (ensures fresh build with latest code)

In [1]:
import subprocess
import os

# Change to workspace directory
os.chdir("/workspaces/updated-minifigures-webshop-2026-LorenzSF")

# Build Docker image
print("Building Docker image...")
result = subprocess.run(
    [
        "docker",
        "build",
        "--no-cache",
        "-t",
        "516454187396.dkr.ecr.eu-west-3.amazonaws.com/frescalorenzo:latest",
        ".",
    ],
    capture_output=False,
    text=True,
)

if result.returncode == 0:
    print("✓ Docker image built successfully!")
else:
    print("✗ Docker build failed!")

Building Docker image...


#0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile:
#1 transferring dockerfile: 4.47kB 0.2s done
#1 DONE 0.3s

#2 resolve image config for docker-image://docker.io/docker/dockerfile:1
#2 ...

#3 [auth] docker/dockerfile:pull token for registry-1.docker.io
#3 DONE 0.0s

#2 resolve image config for docker-image://docker.io/docker/dockerfile:1
#2 DONE 2.4s

#4 docker-image://docker.io/docker/dockerfile:1@sha256:4a43a54dd1fedceb30ba47e76cfcf2b47304f4161c0caeac2db1c61804ea3c91
#4 resolve docker.io/docker/dockerfile:1@sha256:4a43a54dd1fedceb30ba47e76cfcf2b47304f4161c0caeac2db1c61804ea3c91 0.1s done
#4 CACHED

#5 [internal] load metadata for ghcr.io/astral-sh/uv:latest
#5 ...

#6 [auth] library/python:pull token for registry-1.docker.io
#6 DONE 0.0s

#7 [internal] load metadata for docker.io/library/python:3.10-slim
#7 DONE 0.9s

#5 [internal] load metadata for ghcr.io/astral-sh/uv:latest
#5 DONE 1.0s

#8 [in

✓ Docker image built successfully!


## Step 2: Push Docker Image to ECR

Push the built image to your ECR repository

In [ ]:
import subprocess

# Push image to ECR
print("Pushing Docker image to ECR...")
result = subprocess.run(
    ["docker", "push", "516454187396.dkr.ecr.eu-west-3.amazonaws.com/frescalorenzo:latest"],
    capture_output=False,
    text=True,
)

if result.returncode == 0:
    print("✓ Image pushed to ECR successfully!")
else:
    print("✗ Docker push failed!")

## Step 3: SSH into EC2 Instance

Open your terminal and run this command to connect to your EC2 instance:

```bash
ssh ec2-user@35.180.39.26
```

Then become root:
```bash
sudo su
```

Once you're on the EC2 instance (you should see `[root@ip-...]#`), proceed to the next step.

## Step 4: Complete EC2 Deployment Script

Copy and paste the following entire script into your EC2 terminal. This will execute all deployment steps at once:

```bash
#!/bin/bash
set -e

echo "=== Step 1: Stop old containers ==="
docker stop api app 2>/dev/null || true

echo "=== Step 2: Remove old containers ==="
docker rm api app 2>/dev/null || true

echo "=== Step 3: Pull new image ==="
docker pull 516454187396.dkr.ecr.eu-west-3.amazonaws.com/frescalorenzo:latest

echo "=== Step 4: Run API container ==="
docker run -d --network kulroai-net --name api \
  -v $(pwd)/models:/workspaces/minifigures-app/data/models \
  -p 8000:8000 \
  -e PYTHONPATH=/workspaces/minifigures-app/src:$PYTHONPATH \
  516454187396.dkr.ecr.eu-west-3.amazonaws.com/frescalorenzo:latest api

echo "=== Step 5: Run Streamlit app container ==="
docker run -d --network kulroai-net --name app \
  -p 80:8500 \
  -e API_HOST=http://api \
  -e API_PORT=8000 \
  516454187396.dkr.ecr.eu-west-3.amazonaws.com/frescalorenzo:latest app

echo "=== Step 6: Check container status ==="
docker ps

echo "✓ Deployment complete!"
```

## Step 5: Verify Deployment

Run this command on your EC2 instance to check container logs:

```bash
# Check API logs
docker logs api

# Check app logs
docker logs app

# List running containers
docker ps
```

Then open your browser to access the application:
```
http://35.180.39.26
```

If successful, you should see the Minifigures app with Home, Product, and Market pages.

## Summary

**Deployment Flow:**
1. ✓ Build Docker image locally with fresh code
2. ✓ Push image to AWS ECR
3. ✓ SSH into EC2 instance
4. ✓ Stop and remove old containers
5. ✓ Pull new image from ECR
6. ✓ Run API container with PYTHONPATH and environment variables
7. ✓ Run Streamlit app container with API configuration
8. ✓ Access application via IP or domain

**Key Environment Variables:**
- `PYTHONPATH=/workspaces/minifigures-app/src:$PYTHONPATH` - Makes Python find the src modules
- `API_HOST=http://api` - Docker network container name
- `API_PORT=8000` - API port

**Access Points:**
- IP: `http://35.180.39.26`
- Domain: `https://frescalorenzo.realization-of-ai.com`